# Photo to Gaussian Splatting (.ply) in Colab

This notebook combines ideas from two selected candidates:

1. YassGan/3DGaussianSplatting-INRIA-Method-Colab  
   https://github.com/YassGan/3DGaussianSplatting-INRIA-Method-Colab
2. abolhoseinisina/Colab_COLMAP_3DGS  
   https://github.com/abolhoseinisina/Colab_COLMAP_3DGS

Pipeline:
- Input: photos in Google Drive
- COLMAP reconstruction (optional section if you already have COLMAP output)
- 3D Gaussian Splatting training
- Output: `point_cloud.ply`


## 0) Configuration

Set runtime to GPU in Colab: `Runtime -> Change runtime type -> T4 GPU`.

If you already have an undistorted COLMAP dataset, set `RUN_COLMAP = False` and place it under `COLMAP_UNDISTORTED_DIR`.

In [ ]:
RUN_COLMAP = True
IMAGE_SOURCE_DIR = "/content/drive/MyDrive/image_source"
PROJECT_DIR = "/content/project"
COLMAP_UNDISTORTED_DIR = f"{PROJECT_DIR}/undistorted"

# COLMAP feature extraction option:
# 1 if all photos are from one camera, 0 for mixed cameras.
SINGLE_CAMERA = 1

# 3DGS training options
MAX_IMAGE_RESOLUTION = 1600
TRAIN_ITERS = 30000
SAVE_ITERS = "7000 30000"

print("RUN_COLMAP:", RUN_COLMAP)
print("IMAGE_SOURCE_DIR:", IMAGE_SOURCE_DIR)
print("COLMAP_UNDISTORTED_DIR:", COLMAP_UNDISTORTED_DIR)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1) (Optional) Install and run COLMAP from photos

This section follows the COLMAP-focused notebook from candidate #4.

Skip this section if `RUN_COLMAP = False`.

In [ ]:
if RUN_COLMAP:
    !apt-get update
    !apt-get install -y \
      build-essential cmake git \
      libboost-all-dev libeigen3-dev libsuitesparse-dev \
      qtbase5-dev libglew-dev libglfw3-dev \
      libx11-dev libopencv-dev libgoogle-glog-dev \
      libgflags-dev libatlas-base-dev libopencv-core-dev \
      libopenimageio-dev openimageio-tools libopenexr-dev \
      libcgal-dev libcgal-qt5-dev libmetis-dev

In [ ]:
if RUN_COLMAP:
    %cd /content
    !rm -rf abseil-cpp
    !git clone https://github.com/abseil/abseil-cpp.git
    %cd /content/abseil-cpp
    !git checkout 20230802.0
    !rm -rf build && mkdir build
    %cd /content/abseil-cpp/build
    !cmake .. -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX=/usr/local -DCMAKE_POSITION_INDEPENDENT_CODE=ON
    !make -j$(nproc)
    !make install


In [ ]:
if RUN_COLMAP:
    %cd /content
    !rm -rf ceres-solver
    !git clone https://github.com/ceres-solver/ceres-solver.git
    %cd /content/ceres-solver
    !git checkout 2.1.0
    !rm -rf build && mkdir build
    %cd /content/ceres-solver/build
    !cmake .. -DBUILD_TESTING=OFF -DBUILD_EXAMPLES=OFF -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX=/usr/local
    !make -j$(nproc)
    !make install


In [ ]:
if RUN_COLMAP:
    %cd /content
    !rm -rf colmap
    !git clone https://github.com/colmap/colmap.git
    %cd /content/colmap
    !rm -rf build && mkdir build
    %cd /content/colmap/build
    !cmake .. -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX=/usr/local -DCUDA_ENABLED=OFF -DCeres_DIR=/usr/local/lib/cmake/Ceres -DAbsl_DIR=/usr/local/lib/cmake/absl
    !make -j$(nproc)
    !make install
    !colmap --help | head -n 20


In [ ]:
if RUN_COLMAP:
    %cd /content
    !rm -rf {PROJECT_DIR}
    !mkdir -p {PROJECT_DIR}/images
    !cp -r "{IMAGE_SOURCE_DIR}/." "{PROJECT_DIR}/images"

    !colmap feature_extractor \
      --database_path {PROJECT_DIR}/database.db \
      --image_path {PROJECT_DIR}/images \
      --ImageReader.single_camera {SINGLE_CAMERA} \
      --FeatureExtraction.use_gpu 0

    !colmap exhaustive_matcher \
      --database_path {PROJECT_DIR}/database.db \
      --FeatureMatching.use_gpu 0

    !mkdir -p {PROJECT_DIR}/sparse
    !colmap mapper \
      --database_path {PROJECT_DIR}/database.db \
      --image_path {PROJECT_DIR}/images \
      --output_path {PROJECT_DIR}/sparse

    !colmap image_undistorter \
      --image_path {PROJECT_DIR}/images \
      --input_path {PROJECT_DIR}/sparse/0 \
      --output_path {COLMAP_UNDISTORTED_DIR} \
      --output_type COLMAP \
      --max_image_size {MAX_IMAGE_RESOLUTION}

    !mkdir -p "{COLMAP_UNDISTORTED_DIR}/sparse/0"
    !mv {COLMAP_UNDISTORTED_DIR}/sparse/*.bin {COLMAP_UNDISTORTED_DIR}/sparse/0/ || true

    print("COLMAP outputs ready at:", COLMAP_UNDISTORTED_DIR)


## 2) Install 3DGS environment (Colab compatibility)

This section follows candidate #2 style for Colab compatibility.

In [ ]:
!wget -O mini.sh https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh
!chmod +x mini.sh
!bash ./mini.sh -b -f -p /usr/local
!conda install -q -y python=3.7
import sys
sys.path.append('/usr/local/lib/python3.7/site-packages')
!python --version


In [ ]:
!wget -q https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
!chmod +x cuda_11.8.0_520.61.05_linux.run
!./cuda_11.8.0_520.61.05_linux.run --silent --toolkit --no-drm --no-man-page
import os
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.8/lib64:/usr/lib64-nvidia'
!nvcc --version


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==1.12.1+cu116 torchvision==0.13.1+cu116 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu116

import torch
print("torch.cuda.is_available:", torch.cuda.is_available())
print("torch.version.cuda:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
!nvidia-smi


In [ ]:
%cd /content
!rm -rf gaussian-splatting
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!pip install -q plyfile
%cd /content/gaussian-splatting
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q /content/gaussian-splatting/submodules/simple-knn


## 3) Train Gaussian Splatting

Expected dataset structure for `COLMAP_UNDISTORTED_DIR`:

```
undistorted/
  images/
  sparse/0/
    cameras.bin
    images.bin
    points3D.bin
```


In [ ]:
%cd /content/gaussian-splatting
!python train.py -s {COLMAP_UNDISTORTED_DIR} --iterations {TRAIN_ITERS} --save_iterations {SAVE_ITERS}


In [ ]:
# Find newest model dir and check generated .ply files
import os, glob

model_dirs = [d for d in glob.glob('/content/gaussian-splatting/output/*') if os.path.isdir(d)]
model_dirs = sorted(model_dirs, key=lambda p: os.path.getmtime(p))
assert model_dirs, "No model directory found in /content/gaussian-splatting/output"
MODEL_DIR = model_dirs[-1]

ply_files = glob.glob(os.path.join(MODEL_DIR, '**', '*.ply'), recursive=True)
print('MODEL_DIR:', MODEL_DIR)
print('PLY files:')
for p in ply_files:
    print(' -', p)

assert ply_files, 'No .ply file found. Check training logs.'
POINT_CLOUD_PLY = sorted(ply_files, key=lambda p: os.path.getmtime(p))[-1]
print('Selected POINT_CLOUD_PLY:', POINT_CLOUD_PLY)


In [ ]:
# Optional: render train/test views (replace with your actual model dir if needed)
%cd /content/gaussian-splatting
!python render.py -s {COLMAP_UNDISTORTED_DIR} -m {MODEL_DIR}


In [ ]:
# Copy output to Google Drive
EXPORT_DIR = '/content/drive/MyDrive/gaussian_splatting_output'
!mkdir -p {EXPORT_DIR}
!cp -r {MODEL_DIR} {EXPORT_DIR}/
print('Copied model folder to:', EXPORT_DIR)
print('Main ply:', POINT_CLOUD_PLY)


## Notes

- If training crashes because of memory, try fewer input images or lower image resolution in COLMAP undistortion.
- If your COLMAP scene has no `/sparse/0`, inspect mapper logs and input image quality/overlap.
- If you already have COLMAP-ready data, set `RUN_COLMAP = False` and point `COLMAP_UNDISTORTED_DIR` to that dataset.
- The expected final asset is `point_cloud.ply` inside the selected model directory.